In [40]:
import subprocess

import numpy as np
from ase.io import write
from ase.io import read
from ase.db import connect

### Check contents

In [41]:
#db_path = "/LARGE0/gr10563/kai/scripts/abn2db/test/dft_data.db"
#db_path = "/LARGE0/gr10563/kai/scripts/abn2db/test/OUTCAR_01.db"
#db_path = "/LARGE0/gr10563/kai/scripts/abn2db/test/OUTCAR_otf_1k_step.db"

In [42]:
def show_contents(row, column_name=None):
    if column_name:
        if column_name in row:
            print(f"{column_name}: {row[column_name]}")
        else:
            print(f"Column '{column_name}' not found in the row.")
    else:
        for key in row:
            if key in ["positions", "forces"]:
                print(f"{key}: {row[key][0]}")
            else:
                print(f"{key}: {row[key]}")

In [43]:
#db_path = "/LARGE0/gr10563/kai/scripts/abn2db/test/job/results.db"
db_path = "/LARGE0/gr10563/kai/scripts/abn2db/test/job/OUTCAR_otf_1k_step.db"
db = connect(db_path)

### Generate POSCAR

In [44]:
row_count = db.count()
print(f"Number of rows in the database: {row_count}\n")

# Get the first row
row = db.get(1)

show_contents(row)

#print(row.positions)

#row.cell

# Create a POSCAR file from the first_row
#atoms = row.toatoms()
#write('POSCAR', atoms)

Number of rows in the database: 3

sys_name: Ce O
ionic_step: 1
file_name: OUTCAR_otf_1k_step
is_dft: True
common_id: 1
id: 1
unique_id: 518e3f167d47dc3cb03847a25e368c85
ctime: 25.07179142409604
mtime: 25.076622459464282
user: b37873
numbers: [58 58 58 58 58 58 58 58  8  8  8  8  8  8  8  8  8  8  8  8  8  8  8  8]
positions: [0.      1.93747 1.37   ]
cell: [[ 7.74989032  0.          0.        ]
 [ 0.          7.74989032  0.        ]
 [ 0.          0.         20.48      ]]
pbc: [ True  True  True]
calculator: vasp
calculator_parameters: {}
energy: -176.5044534
free_energy: -176.53455746
forces: [-2.496000e-03  4.209000e-03  5.328734e+00]
stress: [-1.667489e+01 -1.667735e+01 -4.292707e+01 -7.400000e-04 -3.330000e-03
  3.100000e-03]


In [45]:
#row = db.get(id=2)
#show_contents(row)

### Check id

In [46]:
"""
db_1 = connect("/LARGE0/gr10563/kai/scripts/abn2db/test/OUTCAR_00.db")
db_2 = connect("/LARGE0/gr10563/kai/scripts/abn2db/test/OUTCAR_01.db")
db_3 = connect("/LARGE0/gr10563/kai/scripts/abn2db/test/merged_OUTCAR_00.db")
db_list = [db_1, db_2, db_3]

for i, db in enumerate(db_list):
    print(f"Database: db_{i+1}")
    for row in db.select():
        show_contents(row, "id")
"""

'\ndb_1 = connect("/LARGE0/gr10563/kai/scripts/abn2db/test/OUTCAR_00.db")\ndb_2 = connect("/LARGE0/gr10563/kai/scripts/abn2db/test/OUTCAR_01.db")\ndb_3 = connect("/LARGE0/gr10563/kai/scripts/abn2db/test/merged_OUTCAR_00.db")\ndb_list = [db_1, db_2, db_3]\n\nfor i, db in enumerate(db_list):\n    print(f"Database: db_{i+1}")\n    for row in db.select():\n        show_contents(row, "id")\n'

In [47]:
"""
# 新しい値を設定するための関数
def set_common_id(db, row_id, common_id_value):
    db.update(row_id, common_id=common_id_value)

# 例として、最初の row に common_id を設定
set_common_id(db_1, 1, 1001)

# 設定が反映されたか確認
row = db_1.get(1)
show_contents(row)
"""

'\n# 新しい値を設定するための関数\ndef set_common_id(db, row_id, common_id_value):\n    db.update(row_id, common_id=common_id_value)\n\n# 例として、最初の row に common_id を設定\nset_common_id(db_1, 1, 1001)\n\n# 設定が反映されたか確認\nrow = db_1.get(1)\nshow_contents(row)\n'

### Check results from execute_vasp.py

In [48]:
dft_db = connect("/LARGE0/gr10563/kai/scripts/abn2db/test/job/OUTCAR_otf_1k_step.db")
mlff_db = connect("/LARGE0/gr10563/kai/scripts/abn2db/test/job/results.db")

In [49]:
for dft_row, mlff_row in zip(dft_db.select(), mlff_db.select()):
    dft_positions = np.array(dft_row.positions)
    mlff_positions = np.array(mlff_row.positions)

    print(f"DFT positions (first row): {dft_positions[0]}")
    print(f"MLFF positions (first row): {mlff_positions[0]}")

    position_diff = np.linalg.norm(dft_positions - mlff_positions, axis=1)
    print(f"Position differences: {position_diff}\n")

Original DFT positions (first row): [0.      1.93747 1.37   ]
Original MLFF positions (first row): [0.      1.93747 1.37   ]
Position differences: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

Original DFT positions (first row): [7.74898 1.93766 1.36841]
Original MLFF positions (first row): [7.74898 1.93766 1.36841]
Position differences: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

Original DFT positions (first row): [7.74827 1.93785 1.36746]
Original MLFF positions (first row): [7.74827 1.93785 1.36746]
Position differences: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]



In [60]:
io_path = "/LARGE0/gr10563/kai/scripts/abn2db/test/job/io_files"
poscar_list = ["POSCAR_0001", "POSCAR_0002", "POSCAR_0003"]

for dft_row, mlff_row, poscar in zip(dft_db.select(), mlff_db.select(), poscar_list):
    poscar_path = f"{io_path}/{poscar}"
    atoms = read(poscar_path)

    dft_positions = np.array(dft_row.positions)
    mlff_positions = np.array(mlff_row.positions)
    atoms_positions = atoms.get_positions()

    print(f"   DFT positions (first row): {dft_positions[0]}")
    print(f"  MLFF positions (first row): {mlff_positions[0]}")
    print(f"POSCAR positions (first row): {atoms_positions[0]} (from {poscar})")

    dft_position_diff = np.linalg.norm(dft_positions - atoms_positions, axis=1)
    mlff_position_diff = np.linalg.norm(mlff_positions - atoms_positions, axis=1)
    
    print(f" Position differences (DFT - OUTCAR): {dft_position_diff}")
    print(f"Position differences (MLFF - OUTCAR): {mlff_position_diff}\n")


   DFT positions (first row): [0.      1.93747 1.37   ]
  MLFF positions (first row): [0.      1.93747 1.37   ]
POSCAR positions (first row): [0.      1.93747 1.37   ] (from POSCAR_0001)
 Position differences (DFT - OUTCAR): [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Position differences (MLFF - OUTCAR): [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

   DFT positions (first row): [7.74898 1.93766 1.36841]
  MLFF positions (first row): [7.74898 1.93766 1.36841]
POSCAR positions (first row): [7.74898 1.93766 1.36841] (from POSCAR_0002)
 Position differences (DFT - OUTCAR): [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Position differences (MLFF - OUTCAR): [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

   DFT positions (first row): [7.74827 1.93785 1.36746]
  MLFF positions (first row): [7.74827 1.93785 1.36746]
POSCAR positions (first row): [7.74827 1.93785 1.36746] (from PO

In [58]:
io_path = "/LARGE0/gr10563/kai/scripts/abn2db/test/job/io_files"
outcar_list = ["OUTCAR_0001", "OUTCAR_0002", "OUTCAR_0003"]

for dft_row, mlff_row, outcar in zip(dft_db.select(), mlff_db.select(), outcar_list):
    outcar_path = f"{io_path}/{outcar}"
    # Extract energy from OUTCAR file
    result = subprocess.run(['grep', 'energy  without entropy=', outcar_path], stdout=subprocess.PIPE)
    energy_line = result.stdout.decode('utf-8').strip()
    energy_value = float(energy_line.split()[4])

    # Extract energy from dft_row and mlff_row
    dft_energy = dft_row.energy
    mlff_energy = mlff_row.energy

    # Compare energies
    energy_diff_dft = abs(dft_energy - energy_value)
    energy_diff_mlff = abs(mlff_energy - energy_value)

    print(f"   DFT energy: {dft_energy}")
    print(f"  MLFF energy: {mlff_energy}")
    print(f"OUTCAR energy: {energy_value} (from {outcar})")
    print(f" Energy difference (DFT - OUTCAR): {energy_diff_dft}")
    print(f"Energy difference (MLFF - OUTCAR): {energy_diff_mlff}\n")

   DFT energy: -176.5044534
  MLFF energy: -175.58324539
OUTCAR energy: -175.58324539 (from OUTCAR_0001)
 Energy difference (DFT - OUTCAR): 0.9212080099999866
Energy difference (MLFF - OUTCAR): 0.0

   DFT energy: -176.49467688
  MLFF energy: -175.57078682
OUTCAR energy: -175.57078682 (from OUTCAR_0002)
 Energy difference (DFT - OUTCAR): 0.9238900599999909
Energy difference (MLFF - OUTCAR): 0.0

   DFT energy: -176.50719818
  MLFF energy: -175.58768273
OUTCAR energy: -175.58768273 (from OUTCAR_0003)
 Energy difference (DFT - OUTCAR): 0.9195154499999774
Energy difference (MLFF - OUTCAR): 0.0

